In [1]:
import matplotlib.pyplot as plt
from matplotlib import font_manager, rc
import platform

# 운영체제별 폰트 설정
if platform.system() == 'Windows':
    # 윈도우 기본 굴림 폰트
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    # 맥(Mac) 기본 폰트 (애플고딕)
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    # 리눅스(Ubuntu 등) 환경에서 나눔고딕이 설치된 경우
    plt.rcParams['font.family'] = 'NanumGothic'

# 마이너스 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

# Import

In [4]:
import os, gc, time, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import spearmanr, norm
from sklearn.base import clone, BaseEstimator
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import BaseCrossValidator, ParameterSampler
import lightgbm as lgb

warnings.filterwarnings("ignore")

from pathlib import Path

In [5]:
# ---------------- 프로젝트 루트 (실행 위치 무관) ----------------
def _find_root():
    here = Path.cwd().resolve()
    for c in [here, *here.parents]:
        if (c / "data").is_dir() and (c / "notebooks").is_dir():
            return c
    return here

PROJECT_ROOT = _find_root()

# ---------------- 수정 ⑳ : src/ 를 import 경로에 ----------------
# model_diagnostics.py 는 notebooks/ 가 아니라 src/ 에 있다.
import sys
for _p in (PROJECT_ROOT / "src", PROJECT_ROOT / "notebooks"):
    if _p.is_dir() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))


# ---------------- parquet 엔진 자동 감지 ----------------
def _pq_engine():
    for name in ("pyarrow", "fastparquet"):
        try:
            __import__(name); return name
        except ImportError:
            continue
    raise ImportError("pyarrow 또는 fastparquet 중 하나가 필요합니다: pip install pyarrow")

PARQUET_ENGINE = _pq_engine()

# ---------------- 설정 ----------------
WINDOWS       = [20, 60, 120, 252]
TRADING_DAYS  = 252
# 수정 ⑩ : 리포 레이아웃(data/processed/)과 구버전(data/) 모두 지원
def _pick(*cands):
    for c in cands:
        if c.exists():
            return c
    return cands[0]

RAW_PATH      = _pick(PROJECT_ROOT / "data/processed/final_df.parquet",
                      PROJECT_ROOT / "data/final_df.parquet")
# CACHE_PATH    = _pick(PROJECT_ROOT / "data/processed/features.parquet",
#                       PROJECT_ROOT / "data/features.parquet")
# v2와 캐시 분리
CACHE_PATH = PROJECT_ROOT / "data/processed/features_v3.parquet"
OUT_DIR       = PROJECT_ROOT / "outputs"
TARGET        = "target_ret_120d"
HORIZON       = 120
MAX_SPAN_DAYS = 200   # shift(-120) 이 달력 200일을 넘으면 데이터 구멍으로 보고 제외

DATE_STRIDE   = 5      # 서치용 날짜 서브샘플 간격 (1 = 전체)
N_ITER        = 20     # RandomizedSearch 시도 횟수
N_SPLITS      = 5
PURGE         = 120    # 타겟 horizon 과 동일해야 함
SELECT_BY     = "daily_ic"   # 수정 ①  ('daily_ic' 또는 'ic_ir')

RUN_XGB       = True
RUN_CAT       = True
RUN_MLP       = False  # sklearn MLP 는 매우 느림. 시간 남을 때만 True

RANDOM_STATE  = 42
COST_BPS      = 10     # 백테스트 왕복 거래비용 (bp)

# ============================================================
# 수정 ⑪~⑬ 설정
# ============================================================
# ⑪ Train/Test 분할일. 계획서 기준은 2023-07-01.
#    None 이면 기존 동작(데이터 끝에서 2년)을 유지한다.
SPLIT_DATE = pd.Timestamp("2023-07-01")

# ⑫ 시점별 S&P500 자격(point-in-time membership) 필터  ★ 중요
#    final_df 에는 "분석기간 중 한 번이라도 구성종목이었던" 티커가 모두 들어있다.
#    각 리밸런싱 날짜의 후보는 '그날 실제 구성종목'이어야 한다.
#    (02_preprocessing.ipynb 이 명시적으로 요구하는 절차 / src/get_tickers.py 참조)
MEMBERSHIP_PATH        = PROJECT_ROOT / "data/raw/cache/sp500_membership_history.csv"
APPLY_MEMBERSHIP_EVAL  = True    # 평가·백테스트 후보를 그날 구성종목으로 제한
APPLY_MEMBERSHIP_TRAIN = False   # 학습셋에도 적용할지. False면 '평가 유니버스만' 바뀌어 A/B가 깨끗하다

# ⑬ 계획서 1차 필터링(안정성 군집) 재현용 스위치.
#    None = 미적용(기존과 동일). 예: ["Stable"], ["Stable", "Moderate"]
STABILITY_FILTER = None

print(f"PROJECT_ROOT   : {PROJECT_ROOT}")
print(f"parquet engine : {PARQUET_ENGINE}")
print(f"설정 완료 | 선정기준={SELECT_BY} | 날짜간격={DATE_STRIDE} | purge={PURGE}")

# ---------------- v3 분석 설정 ----------------

DATA_START = pd.Timestamp("2016-01-01")
DATA_END = pd.Timestamp("2026-06-30")

TRAIN_START = DATA_START
TRAIN_END = pd.Timestamp("2023-06-30")
TEST_START = SPLIT_DATE  # 기존 설정: 2023-07-01
TEST_END = DATA_END

# WINDOWS, TRADING_DAYS, HORIZON은 기존 설정 사용
REBALANCE_DAYS = 120

# 기존 종목 목록 — sector 컬럼은 GICS 기준
UNIVERSE_PATH = PROJECT_ROOT / "data/raw/sp500_universe.csv"

print("룩백:", WINDOWS)
print("리밸런싱:", REBALANCE_DAYS, "거래일")
print("Train:", TRAIN_START.date(), "~", TRAIN_END.date())
print("Test :", TEST_START.date(), "~", TEST_END.date())

PROJECT_ROOT   : /Users/genie/Documents/GitHub/stock-to-rich-Mid-project
parquet engine : pyarrow
설정 완료 | 선정기준=daily_ic | 날짜간격=5 | purge=120
룩백: [20, 60, 120, 252]
리밸런싱: 120 거래일
Train: 2016-01-01 ~ 2023-06-30
Test : 2023-07-01 ~ 2026-06-30


# 원본 가격 + 섹터
final_df

# final_df에서 헬스케어·산업재만 선택
sector_prices

# sector_prices에 지표 16개를 추가한 결과
df_all = add_features(...)

# 군집화 설정

In [12]:
from feature import feature_columns

# ---------------- 군집화 설정 ----------------
TARGET_SECTORS = ["Health Care", "Industrials"]

N_CLUSTERS = 3         # 섹터별 K-Means 군집 수
TOP_N_PER_SECTOR = 10  # 군집 해석 후 섹터별 최종 선정 종목 수

# 위에서 정의한 WINDOWS = [20, 60, 120, 252] 사용
# 룩백: 지표 계산에 사용하는 과거 거래일 수
# feature.py의 실제 지표:
# - beta: 베타
# - volatility: 연율화 변동성
# - return: 과거 기간 수익률 (= v2의 가격 모멘텀)
# - rsi: 상대강도지수
#
# 지표 4가지 × 룩백 4개 = 총 16개 컬럼명
CLUSTER_FEATURES = feature_columns(WINDOWS)

print("타겟 섹터: ", TARGET_SECTORS)
print("군집화 입력 컬럼:", CLUSTER_FEATURES)
print("지표 수:", len(CLUSTER_FEATURES))

# 군집 4개가 안정성·수익성의 4사분면에 해당하는지는 결과로 확인

타겟 섹터:  ['Health Care', 'Industrials']
군집화 입력 컬럼: ['beta_20d', 'volatility_20d', 'return_20d', 'rsi_20d', 'beta_60d', 'volatility_60d', 'return_60d', 'rsi_60d', 'beta_120d', 'volatility_120d', 'return_120d', 'rsi_120d', 'beta_252d', 'volatility_252d', 'return_252d', 'rsi_252d']
지표 수: 16


In [13]:
# ---------------- 기존 CSV의 섹터 정보 연결 ----------------

final_df = pd.read_parquet(RAW_PATH, engine=PARQUET_ENGINE)
universe = pd.read_csv(UNIVERSE_PATH)

# 종목별 섹터 정보
sector_map = (
    universe[["ticker", "sector"]]
    .rename(columns={"ticker": "Ticker"})
    .drop_duplicates()
)

# 한 티커에 여러 섹터가 있으면 확인 후 진행
if sector_map["Ticker"].duplicated().any():
    raise ValueError("같은 티커에 여러 섹터가 등록되어 있습니다.")

# 가격 데이터에 섹터 연결
final_df = (
    final_df.drop(columns=["sector"], errors="ignore")
    .merge(
        sector_map,
        on="Ticker",
        how="left",
        validate="many_to_one",
    )
)

# 헬스케어·산업재만 선택
sector_prices = final_df.loc[
    final_df["sector"].isin(TARGET_SECTORS)
].copy()

print("섹터별 고유 종목 수:")
print(sector_prices.groupby("sector")["Ticker"].nunique())

print("\n섹터 미확인 종목 수:")
print(
    final_df.loc[
        final_df["sector"].isna() | final_df["sector"].eq("Unknown"),
        "Ticker",
    ].nunique()
)

display(sector_prices.head())

섹터별 고유 종목 수:
sector
Health Care    59
Industrials    80
Name: Ticker, dtype: int64

섹터 미확인 종목 수:
209


,Date,Ticker,Open,High,Low,Close,Volume,source,sector
0,2016-01-04,A,37.751921,37.871444,37.089928,37.411728,3287300,yahoo,Health Care
1,2016-01-05,A,37.448503,37.650779,37.089925,37.283005,2587200,yahoo,Health Care
2,2016-01-06,A,36.997989,37.687564,36.823294,37.448509,2103600,yahoo,Health Care
3,2016-01-07,A,36.906044,36.915241,35.683200,35.857891,3504300,yahoo,Health Care
4,2016-01-08,A,36.060167,36.510687,35.370592,35.480923,3736700,yahoo,Health Care


In [14]:
# 종목별로 한 행만 남겨 섹터 연결 상태 확인
ticker_sectors = final_df[["Ticker", "sector"]].drop_duplicates()

unknown = ticker_sectors[
    ticker_sectors["sector"].eq("Unknown")
]
unmatched = ticker_sectors[
    ticker_sectors["sector"].isna()
]

print("CSV에 섹터가 Unknown인 종목:", len(unknown))
print("CSV와 연결되지 않은 종목:", len(unmatched))

display(unknown.head(20))
display(unmatched)

CSV에 섹터가 Unknown인 종목: 209
CSV와 연결되지 않은 종목: 0


,Ticker,sector
2637,AABA,Unknown
3585,AAL,Unknown
6222,AAP,Unknown
14133,ABMD,Unknown
38380,ADT,Unknown
48413,AET,Unknown
51784,AGN,Unknown
55516,AIV,Unknown
71338,ALK,Unknown
79249,ALXN,Unknown


,Ticker,sector


# 군집화용 데이터 준비

In [10]:
# 기존 모듈의 시점별 S&P500 자격 필터 사용
from get_tickers import filter_by_membership

# 실제 필터 적용은 지표 계산 후 진행
# 편입 전 가격도 룩백 지표 계산에 필요하므로 지금은 행을 제거하지 않음

from feature import add_features

# ---------------- 지표 계산 및 캐시 ----------------

BENCHMARK_PATH = PROJECT_ROOT / "data/raw/sp500_beta_df.parquet"

# 최초 실행 또는 데이터·섹터·WINDOWS·feature.py 수정 시 True
# 같은 조건으로 다시 실행할 때는 False로 바꾸면 캐시 재사용
# REBUILD_FEATURES = True
REBUILD_FEATURES = False

if CACHE_PATH.exists() and not REBUILD_FEATURES:
    df_all = pd.read_parquet(
        CACHE_PATH,
        engine=PARQUET_ENGINE,
    )

    required_columns = [
        "Date", "Ticker", "Close", "sector", *CLUSTER_FEATURES
    ]
    missing_columns = [
        col for col in required_columns
        if col not in df_all.columns
    ]

    if missing_columns:
        raise ValueError(
            f"캐시에 필요한 컬럼이 없습니다: {missing_columns}\n"
            "REBUILD_FEATURES = True로 다시 실행하세요."
        )

    print("캐시 로드:", CACHE_PATH)

else:
    t0 = time.time()

    # 앞 셀에서 선택한 헬스케어·산업재 가격 데이터 사용
    prices = sector_prices.copy()
    prices["Date"] = pd.to_datetime(prices["Date"])
    prices = prices.sort_values(["Ticker", "Date"])

    # 베타 계산에 사용할 실제 S&P500 지수 가격
    benchmark = pd.read_parquet(
        BENCHMARK_PATH,
        engine=PARQUET_ENGINE,
    )

    # beta / volatility / return / rsi를 각 룩백으로 계산
    # 편입 전 가격과 초기 결측 행을 유지한 상태에서 계산
    df_all = add_features(
        prices=prices,
        sp500_beta_df=benchmark,
        windows=WINDOWS,
        annualization=TRADING_DAYS,
    )

    CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    df_all.to_parquet(
        CACHE_PATH,
        engine=PARQUET_ENGINE,
        index=False,
    )

    print("캐시 저장:", CACHE_PATH)
    print(f"계산 시간: {time.time() - t0:.1f}초")

df_all["Date"] = pd.to_datetime(df_all["Date"])

print("\n데이터 크기:", df_all.shape)
print("\n지표별 결측 행 수:")
print(df_all[CLUSTER_FEATURES].isna().sum())

display(
    df_all[["Date", "Ticker", "sector", *CLUSTER_FEATURES]].tail()
)

캐시 로드: /Users/genie/Documents/GitHub/stock-to-rich-Mid-project/data/processed/features_v3.parquet

데이터 크기: (351060, 25)

지표별 결측 행 수:
beta_20d            2780
volatility_20d      2780
return_20d          2780
rsi_20d             2780
beta_60d            8304
volatility_60d      8304
return_60d          8304
rsi_60d             8304
beta_120d          16584
volatility_120d    16584
return_120d        16584
rsi_120d           16584
beta_252d          34800
volatility_252d    34800
return_252d        34800
rsi_252d           34800
dtype: int64


,Date,Ticker,sector,beta_20d,volatility_20d,return_20d,rsi_20d,beta_60d,volatility_60d,return_60d,rsi_60d,beta_120d,volatility_120d,return_120d,rsi_120d,beta_252d,volatility_252d,return_252d,rsi_252d
351055,2026-06-24,ZTS,Health Care,0.179858,0.301000,-0.025676,46.221917,0.658537,0.549101,-0.307366,35.892623,0.704060,0.427708,-0.376274,38.757980,0.753394,0.357261,-0.495460,40.655927
351056,2026-06-25,ZTS,Health Care,0.182976,0.299730,-0.020886,46.987439,0.702652,0.545111,-0.328131,34.268887,0.707814,0.427707,-0.376155,38.761887,0.755188,0.357229,-0.499088,40.551336
351057,2026-06-26,ZTS,Health Care,0.225713,0.305158,-0.027852,45.894015,0.689590,0.544133,-0.353515,32.673021,0.709400,0.428515,-0.390508,38.284444,0.756680,0.357774,-0.508541,40.320905
351058,2026-06-29,ZTS,Health Care,0.020995,0.330599,-0.057150,41.746583,0.614026,0.547763,-0.372763,31.849859,0.645953,0.428825,-0.428904,36.656747,0.732079,0.359402,-0.522512,39.980684
351059,2026-06-30,ZTS,Health Care,-0.039525,0.335532,-0.073491,39.672847,0.597617,0.547652,-0.388057,31.021810,0.631164,0.429310,-0.439654,36.300291,0.721772,0.359735,-0.532712,39.710880


In [15]:
# ---------------- 군집화용 데이터 준비 ----------------

# 캐시 원본 df_all은 유지하고 별도 데이터 생성
cluster_data = df_all.loc[
    df_all["Date"].between(DATA_START, DATA_END)
    & df_all["sector"].isin(TARGET_SECTORS)
].copy()

# 1. 지표가 모두 계산된 행만 선택
# 초기 룩백 부족으로 생긴 결측값과 무한대 제외
valid_features = np.isfinite(
    cluster_data[CLUSTER_FEATURES].to_numpy(dtype=float)
).all(axis=1)

n_before = len(cluster_data)
cluster_data = cluster_data.loc[valid_features].copy()

print(f"지표 준비: {n_before:,}행 → {len(cluster_data):,}행")

# 2. 해당 날짜에 실제 S&P500 구성종목인 행만 선택
# 지표 계산이 끝났으므로 이제 편입 전·편출 후 행을 제거해도 됨
n_before = len(cluster_data)

cluster_data = filter_by_membership(
    cluster_data,
    ticker_col="Ticker",
    date_col="Date",
)

cluster_data = (
    cluster_data
    .sort_values(["Date", "Ticker"])
    .reset_index(drop=True)
)

print(f"S&P500 자격 필터: {n_before:,}행 → {len(cluster_data):,}행")

# 3. 군집화용 Train / Test 분리
df_train = cluster_data.loc[
    cluster_data["Date"].between(TRAIN_START, TRAIN_END)
].copy()

df_test = cluster_data.loc[
    cluster_data["Date"].between(TEST_START, TEST_END)
].copy()

# 4. 섹터별 데이터 확인
for name, data in [("Train", df_train), ("Test", df_test)]:
    print(f"\n[{name}]")

    if data.empty:
        raise ValueError(f"{name} 데이터가 없습니다.")

    print(
        "기간:",
        data["Date"].min().date(),
        "~",
        data["Date"].max().date(),
    )

    print(
        data.groupby("sector").agg(
            rows=("Ticker", "size"),
            tickers=("Ticker", "nunique"),
            dates=("Date", "nunique"),
        )
    )

지표 준비: 351,060행 → 316,260행
[캐시] 편입/편출 이력 1259건 재사용 (/Users/genie/Documents/GitHub/stock-to-rich-Mid-project/data/raw/cache/sp500_membership_history.csv)
[INFO] 시점별 자격 미달로 제외된 (종목,날짜) 조합: 34620건 / 316260건
S&P500 자격 필터: 316,260행 → 281,640행

[Train]
기간: 2017-01-03 ~ 2023-06-30
              rows  tickers  dates
sector                            
Health Care  82650       56   1634
Industrials  99914       70   1634

[Test]
기간: 2023-07-03 ~ 2026-06-30
              rows  tickers  dates
sector                            
Health Care  43042       59    751
Industrials  56034       79    751


# 스케일링

In [16]:
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans

# 섹터별 학습 결과 보관
sector_scalers = {}
sector_kmeans = {}
train_clustered = {}

for sector in TARGET_SECTORS:
    # 해당 섹터의 Train 데이터만 사용
    data = df_train.loc[
        df_train["sector"].eq(sector)
    ].copy()

    # 지표마다 다른 단위를 조정
    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(data[CLUSTER_FEATURES])

    # 군집 번호는 우열 순위가 아닌 단순 식별 번호
    model = KMeans(
        n_clusters=N_CLUSTERS,
        random_state=RANDOM_STATE,
        n_init=20,
    )

    data["cluster"] = model.fit_predict(X_scaled)

    sector_scalers[sector] = scaler
    sector_kmeans[sector] = model
    train_clustered[sector] = data

    print(f"\n[{sector}] 군집별 규모")
    display(
        data.groupby("cluster").agg(
            rows=("Ticker", "size"),
            tickers=("Ticker", "nunique"),
        )
    )

    # 스케일링 전 원래 단위로 군집별 지표 평균 확인
    print(f"[{sector}] 군집별 지표 평균")
    display(
        data.groupby("cluster")[CLUSTER_FEATURES]
        .mean()
        .round(3)
    )


[Health Care] 군집별 규모


,rows,tickers
cluster,,
0,11721,56
1,37787,56
2,33142,56


[Health Care] 군집별 지표 평균


,beta_20d,volatility_20d,return_20d,rsi_20d,beta_60d,volatility_60d,return_60d,rsi_60d,beta_120d,volatility_120d,return_120d,rsi_120d,beta_252d,volatility_252d,return_252d,rsi_252d
cluster,,,,,,,,,,,,,,,,
0,1.203,0.493,0.020,53.405,1.168,0.515,0.015,51.413,1.149,0.495,0.025,51.190,1.135,0.446,0.098,51.382
1,0.810,0.210,0.045,61.152,0.826,0.225,0.112,58.997,0.843,0.240,0.185,57.002,0.870,0.260,0.319,55.370
2,0.796,0.260,-0.027,45.147,0.794,0.264,-0.042,47.509,0.799,0.271,-0.041,49.085,0.819,0.278,0.010,50.671



[Industrials] 군집별 규모


,rows,tickers
cluster,,
0,41349,70
1,9386,65
2,49179,70


[Industrials] 군집별 지표 평균


,beta_20d,volatility_20d,return_20d,rsi_20d,beta_60d,volatility_60d,return_60d,rsi_60d,beta_120d,volatility_120d,return_120d,rsi_120d,beta_252d,volatility_252d,return_252d,rsi_252d
cluster,,,,,,,,,,,,,,,,
0,0.988,0.275,-0.027,45.400,0.987,0.275,-0.043,47.475,0.975,0.275,-0.041,49.054,0.964,0.272,0.006,50.561
1,1.400,0.586,0.024,54.256,1.345,0.614,-0.009,50.809,1.287,0.582,-0.069,49.599,1.261,0.502,-0.090,49.865
2,0.984,0.212,0.043,60.301,1.004,0.227,0.110,58.564,1.001,0.244,0.195,57.019,1.013,0.269,0.345,55.541


In [17]:
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from IPython.display import display

# ---------------- 설정 ----------------

N_CLUSTERS = 3
TOP_N_PER_SECTOR = 10

# 기존 feature.py의 실제 컬럼명 사용
CLUSTER_FEATURES = feature_columns(WINDOWS)

VOL_COLS = [f"volatility_{w}d" for w in WINDOWS]
BETA_COLS = [f"beta_{w}d" for w in WINDOWS]
RETURN_COLS = [f"return_{w}d" for w in WINDOWS]
RSI_COLS = [f"rsi_{w}d" for w in WINDOWS]

# Test의 마지막 날짜에서 선정 결과 확인
# 이 날짜 종가까지 계산된 지표를 사용하므로 실제 매매는 이후 거래일 기준
SELECTION_DATE = df_test["Date"].max()

if pd.isna(SELECTION_DATE):
    raise ValueError("Test 데이터가 없습니다.")

sector_scalers = {}
sector_models = {}
sector_cluster_priorities = {}

selected_results = []
cluster_summary_results = []


# ---------------- 섹터별 학습 및 선정 ----------------

for sector in TARGET_SECTORS:
    train = df_train.loc[
        df_train["sector"].eq(sector)
    ].copy()

    candidates = df_test.loc[
        df_test["sector"].eq(sector)
        & df_test["Date"].eq(SELECTION_DATE)
    ].copy()

    if len(train) < N_CLUSTERS:
        raise ValueError(f"{sector}: 군집 학습 데이터가 부족합니다.")

    if candidates["Ticker"].duplicated().any():
        raise ValueError(f"{sector}: 선정일에 티커가 중복되어 있습니다.")

    if len(candidates) < TOP_N_PER_SECTOR:
        raise ValueError(
            f"{sector}: {SELECTION_DATE.date()}의 유효 후보가 "
            f"{len(candidates)}개라 10종목을 선정할 수 없습니다."
        )

    # 1. Train에서만 스케일링 기준 학습
    scaler = RobustScaler()
    X_train = scaler.fit_transform(train[CLUSTER_FEATURES])

    # 2. Train에서만 K-Means 학습
    model = KMeans(
        n_clusters=N_CLUSTERS,
        random_state=RANDOM_STATE,
        n_init=20,
    )

    train["cluster"] = model.fit_predict(X_train)

    if train["cluster"].nunique() != N_CLUSTERS:
        raise ValueError(f"{sector}: 서로 다른 군집 3개가 생성되지 않았습니다.")

    # 3. 군집별 원래 단위의 지표 평균
    # 군집 번호 0/1/2 자체에는 우열 의미가 없음
    centers = train.groupby("cluster")[CLUSTER_FEATURES].mean()

    # 베타는 평균을 낸 뒤 절댓값을 취하지 않고,
    # 각 관측치의 절댓값을 먼저 계산한 뒤 군집별 평균을 사용
    abs_beta_centers = (
        train[BETA_COLS]
        .abs()
        .groupby(train["cluster"])
        .mean()
    )

    # 각 룩백에서 군집 간 순위를 비교한 뒤 평균
    # 낮을수록 상대적으로 안정적인 군집
    volatility_rank = (
        centers[VOL_COLS].rank(ascending=True).mean(axis=1)
    )
    beta_rank = (
        abs_beta_centers.rank(ascending=True).mean(axis=1)
    )

    cluster_risk_score = (
        0.5 * volatility_rank + 0.5 * beta_rank
    )

    cluster_order = (
        cluster_risk_score
        .sort_values(kind="stable")
        .index
        .tolist()
    )

    priority_map = {
        cluster_id: priority
        for priority, cluster_id in enumerate(cluster_order, start=1)
    }

    # 학습 결과 보관
    sector_scalers[sector] = scaler
    sector_models[sector] = model
    sector_cluster_priorities[sector] = priority_map

    # 군집 해석용 요약
    summary = pd.DataFrame({
        "train_rows": train.groupby("cluster").size(),
        "volatility_mean": centers[VOL_COLS].mean(axis=1),
        "abs_beta_mean": abs_beta_centers.mean(axis=1),
        "return_mean": centers[RETURN_COLS].mean(axis=1),
        "rsi_mean": centers[RSI_COLS].mean(axis=1),
        "risk_score": cluster_risk_score,
    })

    summary["priority"] = summary.index.map(priority_map)
    summary["sector"] = sector

    cluster_summary_results.append(summary.reset_index())

    # 4. 선정일 후보에는 Train의 스케일러와 모델을 그대로 적용
    X_candidates = scaler.transform(candidates[CLUSTER_FEATURES])

    candidates["cluster"] = model.predict(X_candidates)
    candidates["cluster_priority"] = (
        candidates["cluster"].map(priority_map)
    )

    # 5. 선정일의 같은 섹터 후보들끼리 순위 점수 계산
    # 모든 점수는 높을수록 우수
    # 각 지표군 내에서는 룩백 4개를 동일하게 반영
    candidates["volatility_score"] = (
        candidates[VOL_COLS]
        .rank(pct=True, ascending=False)
        .mean(axis=1)
    )

    candidates["beta_score"] = (
        candidates[BETA_COLS]
        .abs()
        .rank(pct=True, ascending=False)
        .mean(axis=1)
    )

    candidates["momentum_score"] = (
        candidates[RETURN_COLS]
        .rank(pct=True, ascending=True)
        .mean(axis=1)
    )

    # 높은 RSI를 상승 추세로 해석하는 초기 실험 규칙
    candidates["rsi_score"] = (
        candidates[RSI_COLS]
        .rank(pct=True, ascending=True)
        .mean(axis=1)
    )

    candidates["stability_score"] = (
        candidates["volatility_score"] + candidates["beta_score"]
    ) / 2

    candidates["profitability_score"] = (
        candidates["momentum_score"] + candidates["rsi_score"]
    ) / 2

    candidates["selection_score"] = (
        candidates["stability_score"]
        + candidates["profitability_score"]
    ) / 2

    # 6. 상대적으로 안정적인 군집을 우선
    # 같은 군집에서는 종합 점수가 높은 순서로 선정
    selected = (
        candidates
        .sort_values(
            ["cluster_priority", "selection_score", "Ticker"],
            ascending=[True, False, True],
        )
        .head(TOP_N_PER_SECTOR)
        .copy()
    )

    selected["selection_rank"] = range(1, len(selected) + 1)
    selected_results.append(selected)

    print(f"\n[{sector}] Train 군집 특성")
    print("평균값은 각 지표의 룩백 4개를 단순 평균한 요약입니다.")
    display(
        summary.sort_values("priority").round(3)
    )

    print(f"[{sector}] 선정일 군집별 후보 수")
    display(
        candidates.groupby(
            ["cluster_priority", "cluster"]
        ).size().rename("candidate_count").reset_index()
    )


# ---------------- 최종 결과 ----------------

recommendations = pd.concat(
    selected_results, ignore_index=True
)

cluster_summary = pd.concat(
    cluster_summary_results, ignore_index=True
)

result_columns = [
    "Date",
    "sector",
    "selection_rank",
    "Ticker",
    "cluster",
    "cluster_priority",
    "stability_score",
    "profitability_score",
    "selection_score",
]

print(f"\n선정 기준일: {SELECTION_DATE.date()}")
print(f"최종 선정: {len(recommendations)}종목")

display(
    recommendations[result_columns].round(3)
)

print("\n섹터별 선정 종목 수")
print(recommendations.groupby("sector")["Ticker"].nunique())


[Health Care] Train 군집 특성
평균값은 각 지표의 룩백 4개를 단순 평균한 요약입니다.


,train_rows,volatility_mean,abs_beta_mean,return_mean,rsi_mean,risk_score,priority,sector
cluster,,,,,,,,
1,37787,0.234,0.845,0.165,58.130,1.5,1,Health Care
2,33142,0.268,0.812,-0.025,48.103,1.5,2,Health Care
0,11721,0.487,1.177,0.039,51.847,3.0,3,Health Care


[Health Care] 선정일 군집별 후보 수


,cluster_priority,cluster,candidate_count
0,1,1,20
1,2,2,30
2,3,0,9



[Industrials] Train 군집 특성
평균값은 각 지표의 룩백 4개를 단순 평균한 요약입니다.


...

[Industrials] 선정일 군집별 후보 수


,cluster_priority,cluster,candidate_count
0,1,2,34
1,2,0,33
2,3,1,12



선정 기준일: 2026-06-30
최종 선정: 20종목


,Date,sector,selection_rank,Ticker,cluster,cluster_priority,stability_score,profitability_score,selection_score
0,2026-06-30,Health Care,1,CVS,1,1,0.708,0.900,0.804
1,2026-06-30,Health Care,2,JNJ,1,1,0.784,0.818,0.801
2,2026-06-30,Health Care,3,DGX,1,1,0.839,0.710,0.774
3,2026-06-30,Health Care,4,CAH,1,1,0.731,0.799,0.765
4,2026-06-30,Health Care,5,MRK,1,1,0.731,0.780,0.755
5,2026-06-30,Health Care,6,ABBV,1,1,0.674,0.822,0.748
6,2026-06-30,Health Care,7,DVA,1,1,0.547,0.934,0.740
7,2026-06-30,Health Care,8,LH,1,1,0.811,0.591,0.701
8,2026-06-30,Health Care,9,HSIC,1,1,0.737,0.650,0.694
9,2026-06-30,Health Care,10,BIIB,1,1,0.540,0.814,0.677



섹터별 선정 종목 수
sector
Health Care    10
Industrials    10
Name: Ticker, dtype: int64


In [18]:
# 백테스트 스크립트 실행
# 스크립트는 현재 메모리의 df_all이 아니라 저장된 캐시를 읽습니다.
script_path = PROJECT_ROOT / "scripts" / "backtest_v3_clusters.py"
%run "$script_path"

[캐시] 편입/편출 이력 1259건 재사용 (/Users/genie/Documents/GitHub/stock-to-rich-Mid-project/data/raw/cache/sp500_membership_history.csv)
[INFO] 시점별 자격 미달로 제외된 (종목,날짜) 조합: 34620건 / 316260건
        sector            strategy  days  total_return   cagr  volatility  max_drawdown  sharpe_rf0
Combined_50_50       cluster_top10   720        0.2048 0.0674      0.1280       -0.1383      0.5736
Combined_50_50         score_top10   720        0.2051 0.0675      0.1265       -0.1383      0.5796
Combined_50_50 sector_equal_weight   720        0.3104 0.0992      0.1445       -0.1704      0.7270
   Health Care       cluster_top10   720        0.1806 0.0598      0.1381       -0.1578      0.4897
   Health Care         score_top10   720        0.1774 0.0588      0.1372       -0.1578      0.4853
   Health Care sector_equal_weight   720        0.0948 0.0322      0.1488       -0.1621      0.2875
   Industrials       cluster_top10   720        0.2007 0.0661      0.1529       -0.1976      0.4951
   Industrials         

In [19]:
result_dir = PROJECT_ROOT / "outputs" / "v3_backtest"

# 전략별 성과
summary = pd.read_csv(result_dir / "summary.csv")

display(
    summary.style.format({
        "total_return": "{:.2%}",
        "cagr": "{:.2%}",
        "volatility": "{:.2%}",
        "max_drawdown": "{:.2%}",
        "sharpe_rf0": "{:.3f}",
    })
)

# 군집 전략의 리밸런싱 날짜별 선정 종목
holdings = pd.read_csv(result_dir / "holdings.csv")

display(
    holdings.loc[holdings["strategy"].eq("cluster_top10")]
    .groupby(["signal", "sector"])
    .agg(
        종목수=("ticker", "nunique"),
        선정종목=("ticker", lambda x: ", ".join(x)),
    )
    .reset_index()
)

,sector,strategy,days,total_return,cagr,volatility,max_drawdown,sharpe_rf0
0,Combined_50_50,cluster_top10,720,20.48%,6.74%,12.80%,-13.83%,0.574
1,Combined_50_50,score_top10,720,20.51%,6.75%,12.65%,-13.83%,0.580
2,Combined_50_50,sector_equal_weight,720,31.04%,9.92%,14.45%,-17.04%,0.727
3,Health Care,cluster_top10,720,18.06%,5.98%,13.81%,-15.78%,0.490
4,Health Care,score_top10,720,17.74%,5.88%,13.72%,-15.78%,0.485
5,Health Care,sector_equal_weight,720,9.48%,3.22%,14.88%,-16.21%,0.287
6,Industrials,cluster_top10,720,20.07%,6.61%,15.29%,-19.76%,0.495
7,Industrials,score_top10,720,20.07%,6.61%,15.29%,-19.76%,0.495
8,Industrials,sector_equal_weight,720,54.94%,16.56%,16.47%,-20.24%,1.013


,signal,sector,종목수,선정종목
0,2023-07-03,Health Care,10,"COR, CAH, MCK, ZBH, LLY, MRK, BSX, BDX, HCA, SYK"
1,2023-07-03,Industrials,10,"SNA, TDG, CPRT, AME, PCAR, RSG, PWR, ROL, BR, CSX"
2,2023-12-21,Health Care,10,"COR, ABBV, ABT, LH, REGN, CAH, CVS, AMGN, BSX,..."
3,2023-12-21,Industrials,10,"FAST, BR, PCAR, RSG, ITW, GD, WAB, BA, WM, UNP"
4,2024-06-14,Health Care,10,"MCK, BSX, MRK, COR, LLY, REGN, VRTX, SYK, ELV,..."
5,2024-06-14,Industrials,10,"LDOS, RSG, GD, RTX, VRSK, CTAS, WM, LMT, WAB, HON"
6,2024-12-05,Health Care,10,"BSX, DGX, SYK, COR, CAH, GILD, BMY, DVA, LH, ISRG"
7,2024-12-05,Industrials,10,"BR, PNR, RSG, ADP, WAB, CMI, ITW, SNA, CTAS, AME"
8,2025-06-02,Health Care,10,"CAH, COR, MCK, ABT, HCA, DGX, STE, BSX, GILD, RMD"
9,2025-06-02,Industrials,10,"ROL, RSG, ADP, VRSK, LHX, WM, PAYX, CTAS, BR, RTX"
